In [26]:
import numpy as np
import pandas as pd

from collections import deque

In [27]:
df = pd.read_csv("../data/raw/1993-94.csv")
df.head()

,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27
0,E0,14/08/93,Arsenal,Coventry,0.0,3.0,A,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,E0,14/08/93,Aston Villa,QPR,4.0,1.0,H,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,E0,14/08/93,Chelsea,Blackburn,1.0,2.0,A,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,E0,14/08/93,Liverpool,Sheffield Weds,2.0,0.0,H,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,E0,14/08/93,Man City,Leeds,1.0,1.0,D,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
df.shape

(552, 28)

In [29]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 552 entries, 0 to 551
Data columns (total 28 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Div          462 non-null    str    
 1   Date         462 non-null    str    
 2   HomeTeam     462 non-null    str    
 3   AwayTeam     462 non-null    str    
 4   FTHG         462 non-null    float64
 5   FTAG         462 non-null    float64
 6   FTR          462 non-null    str    
 7   Unnamed: 7   0 non-null      float64
 8   Unnamed: 8   0 non-null      float64
 9   Unnamed: 9   0 non-null      float64
 10  Unnamed: 10  0 non-null      float64
 11  Unnamed: 11  0 non-null      float64
 12  Unnamed: 12  0 non-null      float64
 13  Unnamed: 13  0 non-null      float64
 14  Unnamed: 14  0 non-null      float64
 15  Unnamed: 15  0 non-null      float64
 16  Unnamed: 16  0 non-null      float64
 17  Unnamed: 17  0 non-null      float64
 18  Unnamed: 18  0 non-null      float64
 19  Unnamed: 19  0 non-

In [30]:
df = df.loc[:, ~df.columns.str.startswith("Unnamed")]
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 552 entries, 0 to 551
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Div       462 non-null    str    
 1   Date      462 non-null    str    
 2   HomeTeam  462 non-null    str    
 3   AwayTeam  462 non-null    str    
 4   FTHG      462 non-null    float64
 5   FTAG      462 non-null    float64
 6   FTR       462 non-null    str    
dtypes: float64(2), str(5)
memory usage: 30.3 KB


In [32]:
# Diccionario para guardar historial de cada equipo
historial = {}

# Listas para almacenar resultados
rachas_home = []
rachas_away = []

for idx, row in df.iterrows():
    home = row['HomeTeam']
    away = row['AwayTeam']
    
    # Racha de los últimos 5 (excluyendo partido actual) - como LISTA
    racha_home = historial.get(home, [])[-5:] if home in historial else []
    racha_away = historial.get(away, [])[-5:] if away in historial else []
    
    rachas_home.append(racha_home if racha_home else [])  # Lista vacía si no hay historial
    rachas_away.append(racha_away if racha_away else [])
    
    # Actualizar historial con resultado de ESTE partido
    if row['FTR'] == 'H':
        historial.setdefault(home, []).append('G')
        historial.setdefault(away, []).append('P')
    elif row['FTR'] == 'A':
        historial.setdefault(home, []).append('P')
        historial.setdefault(away, []).append('G')
    else:  # 'D'
        historial.setdefault(home, []).append('E')
        historial.setdefault(away, []).append('E')

# Añadir al DataFrame
df['racha_home'] = rachas_home
df['racha_away'] = rachas_away

In [33]:
df[(df["HomeTeam"] == "Arsenal") | (df["AwayTeam"] == "Arsenal")]

,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,racha_home,racha_away
0,E0,14/08/93,Arsenal,Coventry,0.0,3.0,A,[],[]
11,E0,16/08/93,Tottenham,Arsenal,0.0,1.0,A,[G],[P]
29,E0,21/08/93,Sheffield Weds,Arsenal,0.0,1.0,A,"[P, E]","[P, G]"
34,E0,24/08/93,Arsenal,Leeds,2.0,1.0,H,"[P, G, G]","[E, G, P]"
45,E0,28/08/93,Arsenal,Everton,2.0,0.0,H,"[P, G, G, G]","[G, G, G, P]"
59,E0,01/09/93,Blackburn,Arsenal,1.0,1.0,D,"[G, P, G, G, E]","[P, G, G, G, G]"
66,E0,11/09/93,Arsenal,Ipswich,4.0,0.0,H,"[G, G, G, G, E]","[G, G, P, E, E]"
86,E0,19/09/93,Man United,Arsenal,1.0,0.0,H,"[E, G, G, G, P]","[G, G, G, E, G]"
88,E0,25/09/93,Arsenal,Southampton,1.0,0.0,H,"[G, G, E, G, P]","[G, P, P, P, P]"
101,E0,02/10/93,Liverpool,Arsenal,0.0,0.0,D,"[G, P, P, P, P]","[G, E, G, P, G]"


In [44]:
df.tail(60)

,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,racha_home,racha_away
492,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[E, E, E, E, E]","[E, E, E, E, E]"
493,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[E, E, E, E, E]","[E, E, E, E, E]"
494,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[E, E, E, E, E]","[E, E, E, E, E]"
495,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[E, E, E, E, E]","[E, E, E, E, E]"
496,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[E, E, E, E, E]","[E, E, E, E, E]"
497,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[E, E, E, E, E]","[E, E, E, E, E]"
498,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[E, E, E, E, E]","[E, E, E, E, E]"
499,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[E, E, E, E, E]","[E, E, E, E, E]"
500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[E, E, E, E, E]","[E, E, E, E, E]"
501,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[E, E, E, E, E]","[E, E, E, E, E]"
